# **Step 9 – Consolidación del conocimiento**
**Proyecto TA – Análisis de Datos (1INF03, 2026-2) – Equipo E10**

> *KDD, paso 9 (Fayyad, Piatetsky-Shapiro y Smyth, 1996):* «Consolidating discovered knowledge: incorporating this knowledge into another system for further actions, or documenting it and reporting it to interested parties.»

El conocimiento se consolida de tres formas:
1. **Incorporarlo a un sistema:** una función (`src/atencion.py`) que recalcula el índice cuando llegan meses nuevos de ficha, sin volver a ajustar el modelo.
2. **Entregarlo a la institución:** la lista de prioridad (`step9_lista_prioridad.csv`), la lista de primera infancia (`step9_lista_primera_infancia.csv`) y la tabla de hallazgos con sus acciones.
3. **Documentarlo:** conclusiones, recomendaciones y límites (este notebook, el informe y la presentación).

## Librerías y datos

In [1]:
import warnings; warnings.filterwarnings("ignore")
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from src.atencion import trayectoria, puntajes, cobertura_primera_infancia, PRIORIDAD
LIM, RES = Path("../datos/2_limpios"), Path("../datos/4_resultados")
modelo = json.load(open(RES/"step7_modelo_pca.json"))
p7 = pd.read_csv(RES/"step7_puntajes.csv", sep="|")
c3, c4, c5, c6, c8 = (json.load(open(RES/f"cifras_{k}.json")) for k in ["step3", "step4", "step5", "step6", "step8"])
print("Modelo:", modelo["nombres"], "| varianza:", modelo["varianza_explicada"])

Modelo: ['Permanencia', 'Intensidad psicosocial'] | varianza: [25.47, 18.83]


## 1. Incorporar el conocimiento: recalcular el índice desde la ficha
La función `trayectoria` resume la ficha limpia (niño × mes) y `puntajes` aplica la estandarización y las cargas **guardadas** en el Step 7. Para comprobar que la función reproduce el modelo, se aplica a la ficha actual y se compara con los puntajes del Step 7. Cuando el Puericultorio tenga meses nuevos, basta con limpiar la ficha (Step 3) y llamar a estas dos funciones.

In [2]:
panel = pd.read_csv(LIM/"step3_panel.csv", sep="|")
nuevo = puntajes(trayectoria(panel), modelo)
comp = nuevo.merge(p7[["id_nna", "permanencia", "intensidad", "cuadrante"]], on="id_nna", suffixes=("", "_step7"))
dif = float(np.abs(comp[["permanencia", "intensidad"]].to_numpy() - comp[["permanencia_step7", "intensidad_step7"]].to_numpy()).max())
print(f"Niños: {len(nuevo)} | coinciden con el Step 7: {len(comp)} | diferencia máxima de puntaje: {dif:.1e} | mismo cuadrante: {(comp['cuadrante'] == comp['cuadrante_step7']).mean():.0%}")

Niños: 124 | coinciden con el Step 7: 124 | diferencia máxima de puntaje: 8.9e-16 | mismo cuadrante: 100%


## 2. Entregable para la institución: lista de prioridad
NNA del cuadrante de prioridad, ordenados por índice, con las variables que explican su ubicación y la **estabilidad** (porcentaje de remuestreos en que quedan en ese cuadrante, Step 7), en tres niveles: **1. revisar primero** (en prioridad y estables, ≥ 80 %), **2. revisar** (en prioridad, cerca del borde) y **3. vigilar** (hoy fuera, pero caen en prioridad en la mitad o más de los remuestreos). Los IDs son seudónimos; la lista se revisa con el equipo del Puericultorio.

In [3]:
lista = p7[p7["cuadrante"] == PRIORIDAD].sort_values("indice_prioridad", ascending=False)
lista = lista[["id_nna", "indice_prioridad", "estabilidad_prioridad", "edad", "tiempo_hogar", "upe", "legal_definitiva", "p_sin_soporte", "p_pti_no_tiene", "p_salud_mental"]].copy()
lista["nivel"] = np.where(lista["estabilidad_prioridad"] >= 80, "1. Revisar primero", "2. Revisar")
cols_l = list(lista.columns)
vig = p7[(p7["cuadrante"] != PRIORIDAD) & (p7["estabilidad_prioridad"] >= 50)].sort_values("estabilidad_prioridad", ascending=False)
vig = vig[[c for c in cols_l if c in vig.columns]].assign(nivel="3. Vigilar")
salida_lista = pd.concat([lista, vig], ignore_index=True)
salida_lista.round(2).to_csv(RES/"step9_lista_prioridad.csv", sep="|", index=False)
print(salida_lista["nivel"].value_counts().sort_index().to_string())
lista.head(10).round(2)

nivel
1. Revisar primero    22
2. Revisar            11
3. Vigilar             5


,id_nna,indice_prioridad,estabilidad_prioridad,edad,tiempo_hogar,upe,legal_definitiva,p_sin_soporte,p_pti_no_tiene,p_salud_mental,nivel
0,PPA_FN_0078_M_20110808,3.83,86.00,14.98,170.40,Lima Este,1,1.00,0.89,0.0,1. Revisar primero
1,PPA_FN_0072_M_20080622,3.66,83.67,18.11,150.43,Lima Norte-Callao,1,1.00,1.00,0.0,1. Revisar primero
2,PPA_FN_0039_F_20120610,3.50,81.00,14.14,150.43,Lima Norte-Callao,1,0.95,1.00,0.0,1. Revisar primero
3,PPA_FN_0043_F_20130904,3.18,85.33,12.90,116.69,Lima Este,1,1.00,0.89,0.0,1. Revisar primero
4,PPA_FN_0028_F_20150131,3.16,85.33,11.50,115.51,Lima,1,1.00,0.89,0.0,1. Revisar primero
5,PPA_FN_0064_M_20130128,3.16,85.33,13.50,115.51,Lima,1,1.00,0.89,0.0,1. Revisar primero
6,PPA_FN_0098_M_20061214,3.02,92.67,18.96,190.87,Lima Este,0,0.82,1.00,0.0,1. Revisar primero
7,PPA_FN_0074_M_20070831,2.99,94.67,18.92,163.40,Lima Este,0,1.00,1.00,0.0,1. Revisar primero
8,PPA_FN_0032_F_20121005,2.98,78.00,13.82,149.34,Lima,1,0.79,0.89,0.0,2. Revisar
9,PPA_FN_0050_F_20110813,2.82,96.67,14.97,149.47,Lima Este,0,1.00,1.00,0.0,1. Revisar primero


## 2b. Entregable para la institución: primera infancia
Menores de 6 que, en el último mes de la ficha, llevan meses seguidos **sin estimulación temprana ni educación inicial** (indicador del Step 8, §5). La misma función sirve para actualizar la lista con cada mes nuevo de ficha.

In [4]:
_, pi = cobertura_primera_infancia(panel)
lista_pi = pi[pi["racha_actual"] > 0].sort_values(["racha_actual", "edad"], ascending=False)
lista_pi.round(1).to_csv(RES/"step9_lista_primera_infancia.csv", sep="|", index=False)
lista_pi.round(1)

,id_nna,edad,meses,pct_cubierto,meses_sin_cobertura,racha_actual,upe
14,PPA_FN_0115_F_20220430,4.3,19,73.7,5,5,Lima Este
5,PPA_FN_0006_F_20220530,4.2,19,73.7,5,5,Lima Este
28,PPA_FN_0146_F_20230106,3.6,18,61.1,7,5,Lima Norte-Callao
11,PPA_FN_0103_F_20230309,3.4,19,73.7,5,5,Lima
3,PPA_FN_0004_M_20230901,2.9,19,73.7,5,5,Lima Este


## 3. Hallazgos y acciones

In [5]:
hall = pd.DataFrame([
    {"hallazgo": "La atención psicosocial no aumenta con la permanencia (dos dimensiones independientes)",
     "evidencia": f"PCA: 2 dimensiones reales y estables (Tucker {min(c6['tucker']):.2f}–{max(c6['tucker']):.2f}); {c6['pca_varianza']} % de la varianza",
     "acción sugerida": "Revisar el plan de atención de los niños de larga permanencia"},
    {"hallazgo": f"{c8['cuadrantes'][PRIORIDAD]} niños de larga permanencia con muy poca atención",
     "evidencia": f"{c8['prioridad_pct_sin_soporte']} % sin soporte la mayoría de meses; {c8['prioridad_pct_sin_pti']} % sin PTI; 0 % con salud mental especializada".replace(".", ","),
     "acción sugerida": "Revisar la lista caso por caso, empezando por los 22 más estables (≥ 80 % de remuestreos)"},
    {"hallazgo": "La permanencia difiere entre UPE; la atención no depende de la UPE",
     "evidencia": f"Kruskal-Wallis p < 0,001 (permanencia) y p = {c8['p_intensidad_upe']:.2f} (intensidad)".replace(".", ","),
     "acción sugerida": "Revisar con cada UPE los tiempos de permanencia y la elaboración del PTI"},
    {"hallazgo": "El exceso de peso es generalizado y no depende de la atención",
     "evidencia": f"z de IMC medio +{c4['z_imc_medio']}; ≈ 40 % con exceso en todos los hogares, edades y cuadrantes",
     "acción sugerida": "Intervención nutricional institucional (alimentación, actividad física), no focalizada"},
    {"hallazgo": "La ficha no refleja bien el estado nutricional",
     "evidencia": f"Concordancia ficha–OMS {c5['concordancia']} %; la etiqueta no cambió en 19 meses; cortes más laxos que la OMS",
     "acción sugerida": "Calcular los z OMS de forma automática (src/oms.py) y actualizar la ficha con cada medición"},
    {"hallazgo": f"La primera infancia está cubierta, salvo {len(lista_pi)} niños sin estimulación ni inicial desde marzo de 2026",
     "evidencia": f"{c8['pi_pct_meses_cubiertos']} % de los meses con estimulación (0-2 años) o educación inicial (3-5 años); los huecos se juntan en el paso a inicial (marzo)".replace(".", ","),
     "acción sugerida": "Verificar la matrícula en inicial de los 5 niños y coordinar el paso de la estimulación a inicial antes de marzo"},
    {"hallazgo": "La ficha no registra variables que permitan perfiles de niños",
     "evidencia": "El clustering solo parte en dos la permanencia o repite el estado predominante",
     "acción sugerida": "Registrar motivo de ingreso detallado, desempeño escolar y conducta para análisis futuros"}])
hall.to_csv(RES/"step9_hallazgos.csv", sep="|", index=False); hall

,hallazgo,evidencia,acción sugerida
0,La atención psicosocial no aumenta con la perm...,PCA: 2 dimensiones reales y estables (Tucker 0...,Revisar el plan de atención de los niños de la...
1,33 niños de larga permanencia con muy poca ate...,"97,0 % sin soporte la mayoría de meses; 57,6 %...","Revisar la lista caso por caso, empezando por ..."
2,La permanencia difiere entre UPE; la atención ...,"Kruskal-Wallis p < 0,001 (permanencia) y p = 0...",Revisar con cada UPE los tiempos de permanenci...
3,El exceso de peso es generalizado y no depende...,z de IMC medio +0.98; ≈ 40 % con exceso en tod...,Intervención nutricional institucional (alimen...
4,La ficha no refleja bien el estado nutricional,Concordancia ficha–OMS 68.8 %; la etiqueta no ...,Calcular los z OMS de forma automática (src/om...
5,"La primera infancia está cubierta, salvo 5 niñ...","88,3 % de los meses con estimulación (0-2 años...",Verificar la matrícula en inicial de los 5 niñ...
6,La ficha no registra variables que permitan pe...,El clustering solo parte en dos la permanencia...,"Registrar motivo de ingreso detallado, desempe..."


## 4. Conclusiones
1. **La pregunta del proyecto cambió con la evidencia.** Clasificar la etiqueta nutricional (avance) daba AUC 0,85, pero imitaba un registro manual con rezago. El estado nutricional no forma grupos, y los grupos de la ficha ya estaban escritos en ella. La reducción de dimensiones fue el único método que encontró estructura real, estable y no trivial.
2. **Conocimiento nuevo:** en los NNA de 6 a 19 años, la atención registrada se organiza en **permanencia** e **intensidad psicosocial**, y ambas son **independientes**. Eso deja a más de un cuarto de ellos (33 de 124) con casi 10 años en el hogar y casi ninguna atención.
3. **Conocimiento útil:** una lista de 33 NNA para revisar (22 estables, para revisar primero) y una función para actualizarla con cada mes de ficha.
4. **Primera infancia:** con un indicador propio de su edad, los menores de 6 están cubiertos (88 % de los meses), salvo 5 niños que llevan desde marzo de 2026 sin estimulación ni educación inicial.
5. **Hallazgo complementario:** el exceso de peso (≈ 40 %) es un problema de toda la institución.

**Límites:** 124 NNA de 6 a 19 años de una sola institución (los menores de 6 se evalúan con una regla, no con el PCA); la ficha registra la atención que se da, no las necesidades del niño (un niño sin soporte registrado puede no necesitarlo o no tenerlo anotado); la segunda dimensión es algo menos estable (Tucker p10 = 0,76); las referencias de crecimiento de 5 a 19 años se basan en datos de EE. UU.; el período cubre 19 meses.

**Trabajo futuro:** validar la lista y los nombres de las dimensiones con el equipo del Puericultorio; recalcular el índice cada trimestre; incorporar variables clínicas y escolares a la ficha.

## Verificación y cifras de este notebook

In [6]:
assert dif < 1e-9 and len(comp) == len(p7) and (comp["cuadrante"] == comp["cuadrante_step7"]).all()
assert len(lista_pi) == 5 and len(lista) == 33 and (lista["nivel"] == "1. Revisar primero").sum() == 22 and len(vig) == 5
cifras = {"diferencia_max_funcion": dif, "ninos_lista": len(lista), "revisar_primero": int((lista["nivel"] == "1. Revisar primero").sum()), "vigilar": len(vig), "primera_infancia_sin_cobertura": len(lista_pi)}
json.dump(cifras, open(RES/"cifras_step9.json", "w"), indent=1); cifras

{'diferencia_max_funcion': 8.881784197001252e-16,
 'ninos_lista': 33,
 'revisar_primero': 22,
 'vigilar': 5,
 'primera_infancia_sin_cobertura': 5}